# Bi-Mamba zh↔vi — Notebook xuất biểu đồ báo cáo

Notebook này đọc kết quả của một lần training (`runs/bi_mamba_55m/`) rồi sinh
ra toàn bộ biểu đồ + bảng phục vụ báo cáo:

1. **Đường cong loss** — `train_loss`, `val_loss`, `ema_val_loss` theo step.
2. **Lịch trình learning rate** — cosine + warmup.
3. **Throughput** — token/giây theo step (đo độ ổn định của hardware).
4. **Bảng số tham số** — encoder / decoder / embedding của model đang dùng.
5. **BLEU + chrF** — chạy `evaluate.py` cho từng checkpoint (`latest`, `latest_ema`,
   `best`, `best_ema`, `avg_last5`, `avg_last5_ema`) và một bản ensemble decode
   của 3 checkpoint mạnh nhất, rồi vẽ bar chart so sánh hai chiều dịch.
6. **Một vài câu dịch mẫu** — định tính (qualitative samples).

Tất cả figure được lưu ra `reports/figures/*.png` để paste thẳng vào báo cáo / slide.

> Notebook chỉ cần dữ liệu sinh ra ở cuối lần train: file `metrics.jsonl` (auto)
> và các checkpoint `*.pt` trong `runs/bi_mamba_55m/`. Không cần train lại.


## 1. Mount Drive (tuỳ chọn) + clone repo

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Skipping drive mount:', e)


In [ ]:
import os
REPO_URL = 'https://github.com/ChauDucToan/NLP_DHM.git'
REPO_DIR = '/content/NLP_DHM'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only || true
%cd $REPO_DIR
!pwd && ls


## 2. Cài đặt dependencies

Notebook này chỉ cần `matplotlib` + `pandas` + những gì repo đã yêu cầu.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q matplotlib pandas
import sys; sys.path.insert(0, 'src')
import torch
print('torch:', torch.__version__,
      '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


## 3. Cấu hình đường dẫn

`RUN_DIR` là thư mục mà ô **Train** trong notebook chính (`bi_mamba_zh_vi_colab.ipynb`)
đã ghi checkpoint + `metrics.jsonl` ra. Mặc định là `runs/bi_mamba_55m/`. Nếu bạn
đã copy thư mục này lên Drive, đổi đường dẫn cho phù hợp.

In [ ]:
from pathlib import Path

RUN_DIR    = Path('runs/bi_mamba_55m')               # mặc định: ngay sau train
TOKEN_DIR  = Path('data/tokenizer')                  # spm.model + spm.vocab
CONFIG     = Path('configs/_colab.yaml') if Path('configs/_colab.yaml').exists() else Path('configs/bi_mamba_55m.yaml')
FIG_DIR    = Path('reports/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)

# Optional: ưu tiên dùng Drive nếu đã được copy lên đó.
DRIVE_RUN_DIR = Path('/content/drive/MyDrive/bi_mamba_55m')
if DRIVE_RUN_DIR.exists():
    print(f'Using Drive run dir: {DRIVE_RUN_DIR}')
    RUN_DIR = DRIVE_RUN_DIR
    if (DRIVE_RUN_DIR / 'tokenizer' / 'spm.model').exists():
        TOKEN_DIR = DRIVE_RUN_DIR / 'tokenizer'

print(f'RUN_DIR    = {RUN_DIR}')
print(f'TOKEN_DIR  = {TOKEN_DIR}')
print(f'CONFIG     = {CONFIG}')
print(f'FIG_DIR    = {FIG_DIR}')
print()
print('Files in run dir:')
for p in sorted(RUN_DIR.glob('*'))[:30]:
    print(f'  {p.name:30s} {p.stat().st_size/1e6:.1f} MB')


## 4. Đọc `metrics.jsonl`

Trainer ghi mỗi event 1 dòng JSON ra `<RUN_DIR>/metrics.jsonl`:

* `event=train`: `step`, `loss`, `lr`, `tok/s` — sinh ra mỗi `log_every` step.
* `event=eval`:  `step`, `val_loss`, `best_val_loss`, (optional) `ema_val_loss`, `best_ema_val_loss` — sinh ra mỗi `eval_every` step.


In [ ]:
import json
import pandas as pd

metrics_path = RUN_DIR / 'metrics.jsonl'
assert metrics_path.exists(), (
    f'Không tìm thấy {metrics_path}. Notebook này cần chạy SAU khi train xong. '
    f'Nếu bạn vừa pull một bản train cũ về (chưa có metrics.jsonl), hãy chạy lại '
    f'một segment ngắn để sinh file, hoặc dùng cell parser stdout ở dưới.'
)

records = []
with open(metrics_path, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
df_train = df[df['event'] == 'train'].copy().reset_index(drop=True)
df_eval  = df[df['event'] == 'eval'].copy().reset_index(drop=True)

print(f'Total events: {len(df):,}  |  train: {len(df_train):,}  eval: {len(df_eval):,}')
if len(df_eval) > 0:
    print(f'Last eval @ step {int(df_eval.step.iloc[-1]):,}  |  '
          f'val_loss best={df_eval.best_val_loss.min():.4f}  '
          + (f'ema_val_loss best={df_eval.best_ema_val_loss.min():.4f}' if 'best_ema_val_loss' in df_eval else ''))
df_eval.tail()


### 4.b. (Tuỳ chọn) Parse từ stdout log

Nếu bạn đã train xong nhưng chưa có `metrics.jsonl` (ví dụ chạy bản cũ), hãy
copy stdout của ô Train ra một file `.log` rồi mở comment cell dưới đây và đổi
đường dẫn. Parser lấy regex các dòng `step N | val_loss V (best B) | ema_val_loss V (best B)`.

In [ ]:
# import re
# log_path = '/content/training_stdout.log'   # đổi cho đúng
# rows = []
# pat = re.compile(r'step\s+(\d+)\s+\|\s+val_loss\s+([\d.]+)\s+\(best\s+([\d.]+)\)'
#                  r'(?:\s+\|\s+ema_val_loss\s+([\d.]+)\s+\(best\s+([\d.]+)\))?')
# with open(log_path) as f:
#     for line in f:
#         m = pat.search(line)
#         if m:
#             d = {'event': 'eval', 'step': int(m.group(1)),
#                  'val_loss': float(m.group(2)), 'best_val_loss': float(m.group(3))}
#             if m.group(4):
#                 d['ema_val_loss']      = float(m.group(4))
#                 d['best_ema_val_loss'] = float(m.group(5))
#             rows.append(d)
# import pandas as pd
# df_eval = pd.DataFrame(rows)
# print(df_eval.tail())


## 5. Loss curves (train + val + EMA-val)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 5))

# Train loss — smooth với rolling mean cho dễ đọc.
if len(df_train) > 0:
    smooth = df_train['loss'].rolling(window=max(1, len(df_train)//200), min_periods=1).mean()
    ax.plot(df_train.step, smooth, color='#888', lw=0.9, label='train_loss (smoothed)', alpha=0.9)
# Val loss raw + EMA.
if len(df_eval) > 0:
    ax.plot(df_eval.step, df_eval.val_loss, marker='o', ms=4, color='C0', label='val_loss')
    if 'ema_val_loss' in df_eval:
        ax.plot(df_eval.step, df_eval.ema_val_loss, marker='s', ms=4, color='C1', label='ema_val_loss')
    # Đánh dấu best
    best_step = int(df_eval.loc[df_eval.val_loss.idxmin(), 'step'])
    best_val  = float(df_eval.val_loss.min())
    ax.axvline(best_step, color='C0', ls='--', lw=0.7, alpha=0.5)
    ax.annotate(f'best val_loss={best_val:.3f}\n@ step {best_step:,}',
                xy=(best_step, best_val), xytext=(8, 14), textcoords='offset points',
                fontsize=9, color='C0')
    if 'ema_val_loss' in df_eval:
        bes = int(df_eval.loc[df_eval.ema_val_loss.idxmin(), 'step'])
        bev = float(df_eval.ema_val_loss.min())
        ax.axvline(bes, color='C1', ls='--', lw=0.7, alpha=0.5)
        ax.annotate(f'best ema={bev:.3f}\n@ step {bes:,}',
                    xy=(bes, bev), xytext=(8, -22), textcoords='offset points',
                    fontsize=9, color='C1')

ax.set_xlabel('Training step')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Bi-Mamba zh↔vi — training & validation loss')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', frameon=True)
fig.tight_layout()
out = FIG_DIR / 'loss_curves.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved: {out}')
plt.show()


## 6. Learning-rate schedule

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
if len(df_train) > 0 and 'lr' in df_train.columns:
    ax.plot(df_train.step, df_train.lr, color='C2', lw=1.0)
    ax.set_yscale('log')
    ax.set_xlabel('Training step')
    ax.set_ylabel('Learning rate (log)')
    ax.set_title('LR schedule (warmup → cosine decay)')
    ax.grid(True, which='both', alpha=0.3)
    fig.tight_layout()
    out = FIG_DIR / 'lr_schedule.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved: {out}')
    plt.show()
else:
    print('No LR data in metrics.jsonl')


## 7. Throughput (token / giây)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
if len(df_train) > 0 and 'tok/s' in df_train.columns:
    smooth = df_train['tok/s'].rolling(window=max(1, len(df_train)//100), min_periods=1).mean()
    ax.plot(df_train.step, df_train['tok/s'], color='C3', lw=0.4, alpha=0.4, label='raw')
    ax.plot(df_train.step, smooth, color='C3', lw=1.2, label='smoothed')
    ax.set_xlabel('Training step')
    ax.set_ylabel('Tokens / second (log)')
    ax.set_yscale('log')
    ax.set_title(f'Throughput (mean = {df_train["tok/s"].mean():,.0f} tok/s)')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()
    fig.tight_layout()
    out = FIG_DIR / 'throughput.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved: {out}')
    plt.show()


## 8. Bảng số tham số của model

Phân loại param theo từng khối: embedding (chia sẻ với lm_head), encoder
(N × Bi-Mamba block + FFN), decoder (N × Mamba + cross-attention + FFN).

In [ ]:
import torch
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig

# Lấy cấu hình thực tế từ checkpoint mới nhất (an toàn nếu config hiện tại
# đã đổi so với khi train).
ckpt_for_arch = None
for cand in ['best_ema.pt', 'avg_last5_ema.pt', 'best.pt', 'latest.pt']:
    if (RUN_DIR / cand).exists():
        ckpt_for_arch = RUN_DIR / cand; break
assert ckpt_for_arch is not None, f'Không tìm thấy bất kỳ .pt nào trong {RUN_DIR}'

ckpt = torch.load(ckpt_for_arch, map_location='cpu', weights_only=False)
mcfg = ModelConfig(**ckpt['model_cfg'])
model = BiMambaTranslator(mcfg)

def _sum(prefix):
    return sum(p.numel() for n, p in model.named_parameters() if n.startswith(prefix))

stats = {
    'embedding (tied with lm_head)': _sum('embedding.'),
    'encoder (Bi-Mamba × N)':         _sum('encoder_layers.'),
    'decoder (Mamba + cross-attn × N)': _sum('decoder_layers.'),
    'final norms / heads':            _sum('encoder_norm.') + _sum('decoder_norm.'),
}
total = sum(stats.values())

import pandas as pd
breakdown = pd.DataFrame(
    [(k, v, 100*v/total) for k, v in stats.items()] + [('TOTAL', total, 100.0)],
    columns=['component', 'params', '% of total']
)
breakdown['params'] = breakdown['params'].map(lambda x: f'{x:,}')
print(f'Architecture: d_model={mcfg.d_model}, encoder={mcfg.n_encoder_layers}, '
      f'decoder={mcfg.n_decoder_layers}, d_ff={mcfg.d_ff}, vocab={mcfg.vocab_size}')
breakdown


## 9. BLEU + chrF cho từng checkpoint (+ ensemble decode)

Chạy `scripts/evaluate.py` cho 6 checkpoint và 1 lần ensemble decode 3
checkpoint mạnh nhất. Mỗi lần eval đọc 1000 cặp test mặc định, mất ~5-15 phút
trên T4 / RTX 6000 tuỳ độ dài câu. Kết quả parse được ghi ra
`reports/eval_results.json` để vẽ chart.

In [ ]:
import os, re, json, subprocess

RESULTS_PATH = Path('reports/eval_results.json')
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

# Bạn có thể giảm xuống nếu muốn nhanh hơn.
EVAL_N = 1000

CKPTS = [
    ('latest',         RUN_DIR / 'latest.pt'),
    ('latest_ema',     RUN_DIR / 'latest_ema.pt'),
    ('best',           RUN_DIR / 'best.pt'),
    ('best_ema',       RUN_DIR / 'best_ema.pt'),
    ('avg_last5',      RUN_DIR / 'avg_last5.pt'),
    ('avg_last5_ema',  RUN_DIR / 'avg_last5_ema.pt'),
]
CKPTS = [(n, p) for n, p in CKPTS if p.exists()]

# Ensemble: 3 checkpoint mạnh nhất nếu đủ.
ENSEMBLE_NAMES = ['best_ema', 'avg_last5_ema', 'best']
ensemble_paths = [p for n, p in CKPTS if n in ENSEMBLE_NAMES]
do_ensemble = len(ensemble_paths) >= 2

pat = re.compile(r'\[(\w+)\]\s+n=(\d+)\s+BLEU=([\d.]+)\s+chrF=([\d.]+)\s+\(beam=(\d+),\s*lp=([\d.]+)\)')

def _run(label, args):
    print(f'\n=== {label} ===')
    cmd = ['python', 'scripts/evaluate.py', '--config', str(CONFIG), '--num-samples', str(EVAL_N)] + args
    print('  $', ' '.join(cmd))
    out = subprocess.run(cmd, capture_output=True, text=True)
    print(out.stdout[-1500:])
    if out.returncode != 0:
        print('STDERR:', out.stderr[-1000:])
    parsed = {}
    for m in pat.finditer(out.stdout):
        d, n, bleu, chrf, beam, lp = m.groups()
        parsed[d] = {'n': int(n), 'bleu': float(bleu), 'chrf': float(chrf),
                     'beam': int(beam), 'lp': float(lp)}
    return parsed

results = {}
for name, ckpt in CKPTS:
    results[name] = _run(name, ['--checkpoint', str(ckpt)])

if do_ensemble:
    label = f'ensemble x{len(ensemble_paths)}'
    results[label] = _run(label, ['--checkpoints'] + [str(p) for p in ensemble_paths])

RESULTS_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f'\nSaved: {RESULTS_PATH}')
results


## 10. Bar chart: BLEU + chrF theo checkpoint (cả 2 chiều)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

results = json.loads(RESULTS_PATH.read_text())
order = [k for k in [
    'latest', 'latest_ema',
    'best', 'best_ema',
    'avg_last5', 'avg_last5_ema',
] if k in results] + [k for k in results if k.startswith('ensemble')]

zh2vi_bleu = [results[k].get('zh2vi', {}).get('bleu', None) for k in order]
vi2zh_bleu = [results[k].get('vi2zh', {}).get('bleu', None) for k in order]
zh2vi_chrf = [results[k].get('zh2vi', {}).get('chrf', None) for k in order]
vi2zh_chrf = [results[k].get('vi2zh', {}).get('chrf', None) for k in order]

x = np.arange(len(order))
w = 0.38

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

# BLEU
ax = axes[0]
ax.bar(x - w/2, zh2vi_bleu, w, label='zh→vi', color='C0')
ax.bar(x + w/2, vi2zh_bleu, w, label='vi→zh', color='C3')
for i, v in enumerate(zh2vi_bleu):
    if v is not None: ax.text(i - w/2, v + 0.4, f'{v:.1f}', ha='center', fontsize=8)
for i, v in enumerate(vi2zh_bleu):
    if v is not None: ax.text(i + w/2, v + 0.4, f'{v:.1f}', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=20, ha='right')
ax.set_ylabel('BLEU'); ax.set_title('SacreBLEU (n=1000) — both directions')
ax.grid(True, axis='y', alpha=0.3); ax.legend()

# chrF
ax = axes[1]
ax.bar(x - w/2, zh2vi_chrf, w, label='zh→vi', color='C0')
ax.bar(x + w/2, vi2zh_chrf, w, label='vi→zh', color='C3')
for i, v in enumerate(zh2vi_chrf):
    if v is not None: ax.text(i - w/2, v + 0.4, f'{v:.1f}', ha='center', fontsize=8)
for i, v in enumerate(vi2zh_chrf):
    if v is not None: ax.text(i + w/2, v + 0.4, f'{v:.1f}', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=20, ha='right')
ax.set_ylabel('chrF'); ax.set_title('chrF (n=1000) — both directions')
ax.grid(True, axis='y', alpha=0.3); ax.legend()

fig.tight_layout()
out = FIG_DIR / 'bleu_chrf_per_ckpt.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved: {out}')
plt.show()


## 11. Bảng tóm tắt

Bảng này có thể paste thẳng vào báo cáo / slide.

In [ ]:
rows = []
for name in order:
    r = results.get(name, {})
    rows.append({
        'checkpoint':       name,
        'zh→vi BLEU':       r.get('zh2vi', {}).get('bleu'),
        'zh→vi chrF':       r.get('zh2vi', {}).get('chrf'),
        'vi→zh BLEU':       r.get('vi2zh', {}).get('bleu'),
        'vi→zh chrF':       r.get('vi2zh', {}).get('chrf'),
    })
summary = pd.DataFrame(rows)
summary.to_csv('reports/summary.csv', index=False)
summary


## 12. Vài câu dịch mẫu (qualitative)

In [ ]:
# Lấy checkpoint mạnh nhất hiện có để dịch demo.
import torch
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
from bi_mamba_mt.tokenizer import Tokenizer
from bi_mamba_mt.translate import translate_batch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
demo_pri = ['avg_last5_ema.pt', 'best_ema.pt', 'best.pt', 'latest.pt']
demo_path = next((RUN_DIR / p for p in demo_pri if (RUN_DIR / p).exists()), None)
print(f'Demo checkpoint: {demo_path}')
ckpt = torch.load(demo_path, map_location='cpu', weights_only=False)
m = BiMambaTranslator(ModelConfig(**ckpt['model_cfg'])).to(device); m.load_state_dict(ckpt['model'], strict=False); m.eval()
tok = Tokenizer(TOKEN_DIR / 'spm.model')

demos_zh2vi = [
    '我们今天去吃越南河粉。',
    '人工智能正在改变世界。',
    '她每天早上六点起床去跑步。',
    '机器翻译还有很多需要改进的地方。',
]
demos_vi2zh = [
    'Hôm nay trời rất đẹp.',
    'Trí tuệ nhân tạo đang thay đổi thế giới.',
    'Cô ấy đã đi ngủ từ rất sớm.',
    'Học máy giúp giải quyết nhiều vấn đề khó.',
]

print('\n--- zh → vi ---')
out_zh2vi = translate_batch(m, tok, demos_zh2vi, 'zh2vi', beam_size=6, length_penalty=1.20, device=device)
for s, t in zip(demos_zh2vi, out_zh2vi):
    print(f'  {s}\n    -> {t}')

print('\n--- vi → zh ---')
out_vi2zh = translate_batch(m, tok, demos_vi2zh, 'vi2zh', beam_size=6, length_penalty=0.80, device=device)
for s, t in zip(demos_vi2zh, out_vi2zh):
    print(f'  {s}\n    -> {t}')

# Lưu ra file để paste vào báo cáo.
import json
samples = {
    'zh2vi': [{'src': s, 'hyp': t} for s, t in zip(demos_zh2vi, out_zh2vi)],
    'vi2zh': [{'src': s, 'hyp': t} for s, t in zip(demos_vi2zh, out_vi2zh)],
    'checkpoint': str(demo_path.name),
}
Path('reports/qualitative_samples.json').write_text(json.dumps(samples, indent=2, ensure_ascii=False))
print('\nSaved: reports/qualitative_samples.json')


## 13. Done — files for the report

```
reports/
├── figures/
│   ├── loss_curves.png
│   ├── lr_schedule.png
│   ├── throughput.png
│   └── bleu_chrf_per_ckpt.png
├── eval_results.json          # raw BLEU/chrF per checkpoint
├── summary.csv                # bảng tóm tắt
└── qualitative_samples.json   # vài câu dịch mẫu
```

Paste mấy file PNG này thẳng vào báo cáo / slide. Bảng `summary.csv` mở
trực tiếp với Excel / Google Sheets.
